# TorchPenny `module.py` acceptance tests

수정 후 기대하는 **동작**을 검증합니다. PASS는 해당 동작이 검증되었다는 뜻이며,
FAIL은 테스트 이름 아래의 원인과 함께 표시됩니다. 다른 테스트는 계속 실행됩니다.

1. PyTorch와 PennyLane이 설치된 `torchpenny` 환경을 커널로 선택하세요.
2. `module.py`를 수정한 뒤에는 **Restart Kernel → Run All**을 실행하세요.
   이미 만들어진 인스턴스는 import만 다시 해도 갱신되지 않습니다.
3. 개별 테스트 셀을 다시 실행하면 같은 이름의 결과를 덮어씁니다.
4. 마지막 요약 셀은 누락된 테스트도 표시합니다. 설정 셀에서
   `STRICT_MODE = True`로 바꾸면 실패 또는 누락이 있을 때 assertion을 발생시킵니다.

라이브러리 코드는 이 노트북에서 변경하지 않습니다. `torchpenny.lib`도 import하지 않습니다.
저장된 과거 실행 결과를 지웠으므로 현재 커널에서 다시 실행해 확인하세요.

In [1]:
import sys
from pathlib import Path

# Works when launched from the repository root, test/, or a descendant.
project_root = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "torchpenny" / "module.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start this notebook inside the TorchPenny repository.")

sys.dont_write_bytecode = True
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))

In [2]:
import copy
import inspect

import pennylane as qml
import torch
from torch import nn

import torchpenny.module as core
from torchpenny.module import QLayer, QSubLayer

require_local_path = (project_root / "torchpenny" / "module.py").resolve()
if Path(core.__file__).resolve() != require_local_path:
    raise RuntimeError("Another TorchPenny installation is loaded; restart the kernel.")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("Module:", core.__file__)

Python: 3.12.9
PyTorch: 2.6.0
PennyLane: 0.41.0
Module: /Users/hyunseongkim/Documents/GitHub/TorchPenny/torchpenny/module.py


## Test policy

PyTorch feature 중심 정책을 적용합니다.

- 입력 `(..., F)`에 대해 단일 expectation 출력은 `(..., 1)`입니다.
- batch 없는 단일 expectation은 `(1,)`이며 finite-shot에서도 feature 축을 유지합니다.
- 외부 파라미터 template은 `(1, N)`으로 통일합니다. 실제 batched 입력은 `(B, N)`입니다.

`STRICT_MODE`는 마지막 요약 셀에서 적용됩니다.
wire 검증은 현재 사용자 수정과 호환되도록 `AssertionError`도 허용합니다.
이는 일반 실행에서의 검증이며, Python `-O`에서도 검증이 유지됨을 보장하지는 않습니다.

In [ ]:
STRICT_MODE = False

In [4]:
TEST_RESULTS = {}


def require(condition, message):
    if not condition:
        raise AssertionError(message)


def require_raises(expected_exceptions, action):
    try:
        action()
    except expected_exceptions as exc:
        return exc
    except Exception as exc:
        raise AssertionError(
            f"Expected {expected_exceptions}, received {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError("Expected an exception, but no exception was raised.")


def run_test(name, test_function):
    try:
        test_function()
    except Exception as exc:
        TEST_RESULTS[name] = (False, f"{type(exc).__name__}: {exc}")
        print(f"[FAIL] {name}\n       {type(exc).__name__}: {exc}")
    else:
        TEST_RESULTS[name] = (True, "")
        print(f"[PASS] {name}")


def check_boundary_rejection(layer, invalid):
    # An invalid input must be rejected before any quantum execution.
    def unexpected_execution(*args, **kwargs):
        raise AssertionError("Invalid input reached the QNode.")
    layer.qnode = unexpected_execution
    require_raises(TypeError, lambda: layer(invalid))

## Minimal fixtures

공통 클래스는 이 셀에서 한 번만 정의합니다. 테스트 간 원인 분리를 위해 실행 테스트는
필요한 `interface`와 `diff_method`를 명시하며, 기본 설정 자체는 별도 테스트에서 확인합니다.
`to_operation()`은 이 테스트에서 사용하지 않는 기능입니다.

In [5]:
class ParameterBlock(QSubLayer):
    def init_weights(self):
        return torch.zeros(1, self.wires, dtype=torch.float32)

    def forward(self, x=None, wires=()):
        params = self.get_params(x)
        for index, wire in enumerate(wires):
            angle = params[:, index] if params.ndim > 1 else params[index]
            qml.RY(angle, wires=wire)

    def to_operation(self):
        raise NotImplementedError("Not used by these tests.")


class FlatExternalBlock(QSubLayer):
    def init_weights(self):
        return torch.zeros(self.wires, dtype=torch.float32)

    def forward(self, x=None, wires=()):
        return None

    def to_operation(self):
        raise NotImplementedError("Not used by these tests.")


class ProbabilityLayer(QLayer):
    def inner_gates(self, x):
        qml.RY(x[:, 0], wires=0)

    def measurement(self):
        return qml.probs(wires=range(self.wires))


class ScalarLayer(QLayer):
    def inner_gates(self, x):
        qml.RY(x[:, 0], wires=0)

    def measurement(self):
        return qml.expval(qml.PauliZ(0))


class MixedMeasurementLayer(QLayer):
    def inner_gates(self, x):
        qml.RY(x[:, 0], wires=0)

    def measurement(self):
        return qml.expval(qml.PauliZ(0)), qml.probs(wires=[0])


class SamplingLayer(QLayer):
    def inner_gates(self, x):
        qml.RY(x[:, 0], wires=0)

    def measurement(self):
        return qml.sample(wires=range(self.wires))


class CompositeLayer(QLayer):
    def __init__(self, wires=2):
        super().__init__(wires)
        self.encoding = ParameterBlock(wires, param_received=True)

    def inner_gates(self, x):
        self.encoding(x, wires=range(self.wires))

    def measurement(self):
        return qml.probs(wires=range(self.wires))

class NoInputLayer(QLayer):
    def inner_gates(self):
        qml.Hadamard(wires=0)

    def measurement(self):
        return qml.probs(wires=range(self.wires))


class OwnedParameterLayer(QLayer):
    def __init__(self):
        super().__init__(
            1, qnode_kwargs={"interface": "torch", "diff_method": "backprop"}
        )
        self.block = ParameterBlock(1)
        with torch.no_grad():
            self.block.params.fill_(0.37)

    def inner_gates(self):
        self.block(wires=(0,))

    def measurement(self):
        return qml.expval(qml.PauliZ(0))


## 1. Wire counts must be positive integers

This is the first example: booleans and floating-point values must be rejected instead of silently becoming integers.

In [6]:
def check_wire_validation(factory):
    rejection_errors = (AssertionError, TypeError, ValueError)
    for invalid in (True, False, 1.5, 2.0, "234", "0.23", "ASB", ("as",), None):
        require_raises(rejection_errors, lambda value=invalid: factory(value))
    for invalid in (0, -1):
        require_raises(rejection_errors, lambda value=invalid: factory(value))
    for valid in (1, 2, 4):
        require(factory(valid).wires == valid, f"Wire count {valid} was not retained.")


run_test("QSubLayer wire validation", lambda: check_wire_validation(ParameterBlock))
run_test("QLayer wire validation", lambda: check_wire_validation(NoInputLayer))

[PASS] QSubLayer wire validation
[PASS] QLayer wire validation


## 2. Configuration dictionaries must not be mutated

Constructing a layer must not add `wires` or `interface` to dictionaries owned by the caller or to shared default objects.

In [7]:
def test_configuration_is_not_mutated():
    device_kwargs = {}
    qnode_kwargs = {"diff_method": "backprop"}
    expected_device_kwargs = copy.deepcopy(device_kwargs)
    expected_qnode_kwargs = copy.deepcopy(qnode_kwargs)

    NoInputLayer(
        1,
        q_device_kwargs=device_kwargs,
        qnode_kwargs=qnode_kwargs,
    )

    require(
        device_kwargs == expected_device_kwargs,
        f"q_device_kwargs was mutated to {device_kwargs}.",
    )
    require(
        qnode_kwargs == expected_qnode_kwargs,
        f"qnode_kwargs was mutated to {qnode_kwargs}.",
    )


run_test("Caller-owned configuration remains unchanged", test_configuration_is_not_mutated)

[PASS] Caller-owned configuration remains unchanged


In [8]:
def test_default_configuration_is_not_shared_state():
    parameters = inspect.signature(QLayer.__init__).parameters
    device_default = parameters["q_device_kwargs"].default
    qnode_default = parameters["qnode_kwargs"].default
    before_device = copy.deepcopy(device_default)
    before_qnode = copy.deepcopy(qnode_default)

    NoInputLayer(1)
    NoInputLayer(2)

    require(
        device_default == before_device,
        f"Default q_device_kwargs changed from {before_device} to {device_default}.",
    )
    require(
        qnode_default == before_qnode,
        f"Default qnode_kwargs changed from {before_qnode} to {qnode_default}.",
    )


run_test("Default configuration does not retain instance state", test_default_configuration_is_not_shared_state)

[PASS] Default configuration does not retain instance state


## 3. External shape metadata and parameter ownership

내부·외부 모드 모두 `init_weights()`에서 얻은 shape를 읽기 전용 `parameter_shape`로 보관합니다.

- 내부 모드의 `params`는 등록된 `Parameter`이며 `.to()`와 `state_dict()`를 지원합니다.
- 외부 모드의 `params`는 `None`이며 shape용 tensor를 보관하지 않습니다.
  shape는 device/dtype이 없으므로 변환할 필요가 없습니다. `set_parameters()`는 외부 모드에서 거부합니다.
- 내부 파라미터의 학습 여부는 `requires_grad_()`로 전환하며 값과 shape는 유지합니다.
  외부 입력 ↔ 내부 파라미터 전환 API는 이번 변경에 포함하지 않습니다.

속성 존재 여부만 확인하지 않고 shape, 개수, 등록 상태, 변환 후 일관성을 검증합니다.
외부 입력은 모듈에 등록하지 않으며 gradient가 그대로 흐르는지도 별도로 검사합니다.

In [ ]:
def test_external_parameter_metadata():
    for wires in (1, 2, 4):
        external = ParameterBlock(wires, param_received=True)
        internal = ParameterBlock(wires, param_received=False)
        expected = torch.Size((1, wires))

        require(internal.params.shape == expected, "Internal initialization shape changed.")
        require(external.input_dim == expected, "External input_dim is incorrect.")
        require(external.num_params == internal.num_params == wires, "Parameter counts differ.")
        require(not dict(external.named_parameters()), "External mode owns Parameters.")

        require(external.params is None, "External params must be None.")
        require(not dict(external.named_buffers()), "Shape metadata must not retain a buffer.")
        require(not external.state_dict(), "External metadata must not add checkpoint tensors.")
        require(internal.input_dim is None, "Internal mode must not require external input.")
        for layer in (internal, external):
            require(layer.parameter_shape == expected, "Shared parameter_shape is incorrect.")
            require_raises(AttributeError, lambda: setattr(layer, "parameter_shape", (9,)))
        require_raises(RuntimeError, lambda: external.set_parameters(torch.zeros(expected)))

        external.to(dtype=torch.float64)
        require(external.params is None, "Conversion created external parameters.")
        require(external.parameter_shape == expected, "Conversion changed parameter_shape.")
        require(external.input_dim == expected, "Conversion changed input_dim.")
        require(external.num_params == wires, "Conversion changed the parameter count.")


def test_internal_parameter_state():
    layer = ParameterBlock(2).to(dtype=torch.float64)
    require(
        dict(layer.named_parameters()).get("params") is layer.params,
        "Internal params is not registered.",
    )
    require(layer.params.requires_grad, "Internal params must be trainable.")
    require(layer.params.dtype == torch.float64, "Parameter conversion failed.")
    with torch.no_grad():
        layer.params.fill_(0.37)
    restored = ParameterBlock(2).to(dtype=torch.float64)
    restored.load_state_dict(layer.state_dict())
    torch.testing.assert_close(restored.params, layer.params)
    require(layer.parameter_shape == layer.params.shape, "Internal shape metadata changed.")
    original_parameter = layer.params
    expected_values = layer.params.detach().clone()
    for trainable in (False, True):
        layer.requires_grad_(trainable)
        require(layer.params is original_parameter, "Freezing replaced the Parameter.")
        require(layer.params.requires_grad == trainable, "Trainability did not change.")
        require(layer.parameter_shape == torch.Size((1, 2)), "Freezing changed shape metadata.")
        require(layer.num_params == 2 and layer.input_dim is None, "Freezing changed ownership metadata.")
        torch.testing.assert_close(layer.params, expected_values)


def test_external_input_gradient():
    block = ParameterBlock(1, param_received=True)
    require_raises((AssertionError, TypeError, ValueError), lambda: block.get_params())
    require_raises(
        (AssertionError, TypeError, ValueError),
        lambda: block.get_params(torch.zeros(2, 3)),
    )
    x = torch.tensor([[0.2], [0.7]], dtype=torch.float64, requires_grad=True)
    require(block.get_params(x) is x, "get_params must preserve the external tensor.")
    @qml.qnode(qml.device("default.qubit", wires=1), interface="torch", diff_method="backprop")
    def circuit(data):
        block(data, wires=(0,))
        return qml.expval(qml.PauliZ(0))
    value = circuit(x)
    torch.testing.assert_close(value, torch.cos(x[:, 0]))
    value.sum().backward()
    torch.testing.assert_close(x.grad, -torch.sin(x.detach()))
    require(not dict(block.named_parameters()), "Input was registered as a Parameter.")


run_test("External parameter metadata integrates with PyTorch", test_external_parameter_metadata)
run_test("Internal parameters support conversion and state_dict", test_internal_parameter_state)
run_test("External input preserves values and gradients", test_external_input_gradient)

## 4. Parameter replacement must preserve ownership

`set_parameters()` should retain the registered `Parameter` object and copy values without sharing storage with the caller's tensor.

In [ ]:
def test_safe_parameter_copy():
    layer = ParameterBlock(2, param_received=False)
    original_parameter = layer.params
    source = torch.tensor([[1.0, 2.0]])

    layer.set_parameters(source)

    require(
        layer.params is original_parameter,
        "set_parameters() replaced the registered Parameter object.",
    )
    require(
        torch.equal(layer.params.detach(), torch.tensor([[1.0, 2.0]])),
        "The requested parameter values were not copied.",
    )

    source.add_(10.0)
    require(
        torch.equal(layer.params.detach(), torch.tensor([[1.0, 2.0]])),
        "The module parameter still aliases caller-owned storage.",
    )
    require_raises(TypeError, lambda: layer.set_parameters([[1.0, 2.0]]))
    require_raises(ValueError, lambda: layer.set_parameters(torch.zeros(2)))
    source = torch.tensor([[3.0, 4.0]], dtype=torch.float64, requires_grad=True)
    for trainable in (False, True):
        layer.requires_grad_(trainable)
        layer.set_parameters(source)
        require(layer.params is original_parameter, "Copy replaced the Parameter.")
        require(layer.params.requires_grad == trainable, "Copy changed trainability.")
        require(layer.params.dtype == torch.float32, "Copy changed destination dtype.")
        torch.testing.assert_close(layer.params, source.detach().float())
    layer.params.sum().backward()
    torch.testing.assert_close(layer.params.grad, torch.ones_like(layer.params))
    require(source.grad is None, "Copy retained the source computation graph.")


run_test("set_parameters safely copies values", test_safe_parameter_copy)

## 5. Required subclass methods must be enforced

Subclass that didn't implement `forward()` must be rejected in the creation stage.


`forward()`를 구현하지 않은 서브클래스는 생성 시 거부되어야 합니다.
특정 metaclass 사용 여부를 직접 검사하지 않고 생성 결과를 확인합니다.
`init_weights()`도 필수입니다. `to_operation()`은 선택 기능이므로 미구현이어도 생성할 수 있어야 합니다.

In [ ]:
def test_abstract_contract():
    class MissingForwardBlock(QSubLayer):
        def init_weights(self):
            return torch.zeros(1, self.wires)

        def to_operation(self):
            raise NotImplementedError

    require_raises(TypeError, lambda: MissingForwardBlock(1))
    class MissingWeightsBlock(QSubLayer):
        def forward(self, x=None, wires=()):
            return self.get_params(x)

    require_raises(TypeError, lambda: MissingWeightsBlock(1))
    class MinimalBlock(MissingWeightsBlock):
        def init_weights(self):
            return torch.zeros(1, self.wires)

    minimal = MinimalBlock(1)
    require(minimal.num_params == 1, "Optional to_operation blocked construction.")
    require(minimal() is minimal.params, "Inherited forward is not usable.")
    require_raises(NotImplementedError, minimal.to_operation)
    # A complete subclass must still be usable.
    require(ParameterBlock(1).num_params == 1, "A complete subclass cannot be constructed.")


run_test("QSubLayer enforces required subclass methods", test_abstract_contract)

## 6. Child-module registration must have one source of truth

After replacing a `QSubLayer`, attribute lookup, PyTorch's child registry, and `input_features` must agree.

In [ ]:
def test_sublayer_registry_consistency():
    layer = CompositeLayer(2)
    replacement = nn.Identity()
    layer.encoding = replacement

    require(
        layer.encoding is replacement,
        "Attribute lookup returned the old QSubLayer.",
    )
    require(
        layer._modules["encoding"] is replacement,
        "PyTorch's child registry did not contain the replacement.",
    )

    compatibility_registry = getattr(layer, "_qsublayers", {})
    require(
        "encoding" not in compatibility_registry,
        "The compatibility sublayer registry retained the replaced QSubLayer.",
    )
    require(
        "encoding" not in layer.input_features,
        "input_features retained a replaced external sublayer.",
    )
    external = ParameterBlock(3, param_received=True)
    layer.add_module("encoding", external)
    require(layer.encoding is external, "add_module did not update the child.")
    require(layer.input_features["encoding"] == torch.Size([3]), "New shape was not reported.")
    require_raises(TypeError, lambda: setattr(layer, "encoding", 7))
    require(layer.encoding is external, "Rejected assignment changed the child.")
    layer.encoding = None
    require("encoding" not in layer.input_features, "None retained external metadata.")
    internal = ParameterBlock(2)
    layer._register_qsublayer("encoding", internal)
    require(dict(layer.named_parameters())["encoding.params"] is internal.params,
            "Registration helper bypassed PyTorch.")
    require("encoding.params" in layer.state_dict(), "Registered parameters are missing.")
    require("encoding" not in layer.input_features, "Internal child was reported as external.")


run_test("Sublayer replacement keeps registries consistent", test_sublayer_registry_consistency)

In [ ]:
def test_sublayer_deletion():
    layer = CompositeLayer(2)
    del layer.encoding
    require(not hasattr(layer, "encoding"), "Deleted child remains visible.")
    require("encoding" not in dict(layer.named_children()), "Deleted child remains registered.")
    require("encoding" not in layer.input_features, "Deleted child's metadata remains.")
    layer.encoding = ParameterBlock(2)
    del layer.encoding
    require("encoding.params" not in layer.state_dict(), "Deleted parameter remains in checkpoint.")
    layer.encoding = ParameterBlock(2, param_received=True)
    require(layer.input_features["encoding"] == torch.Size([2]), "Child cannot be re-registered.")


run_test("Sublayer deletion removes stale state", test_sublayer_deletion)

## 7. External parameter templates must have shape (1, N)

외부 모드는 `(1, N)` template을 사용합니다. 다른 shape는 사용자가 `init_weights()`에서 변환해야 합니다.
생성 시 잘못된 shape를 거부하고, 정상 template의 입력 정보와 repr을 검증합니다. 내부 파라미터 shape는 제한하지 않습니다.

In [ ]:
def test_external_shape_contract():
    require_raises(ValueError, lambda: FlatExternalBlock(3, param_received=True))
    for shape in ((), (3,), (2, 3), (1, 2, 3)):
        class ShapedBlock(ParameterBlock):
            def init_weights(self):
                return torch.zeros(shape)

        require_raises(ValueError, lambda: ShapedBlock(3, param_received=True))
        require(ShapedBlock(3).parameter_shape == shape, "Internal shape was restricted.")
    class AdaptedBlock(FlatExternalBlock):
        def init_weights(self):
            return super().init_weights().unsqueeze(0)

    for factory in (ParameterBlock, AdaptedBlock):
        holder = NoInputLayer(3)
        holder.encoding = factory(3, param_received=True)
        require(holder.input_features["encoding"] == torch.Size([3]), "Input shape is incorrect.")
        require(holder.encoding.parameter_shape == torch.Size([1, 3]), "Template shape changed.")


def test_external_repr():
    for wires in (1, 3):
        block = ParameterBlock(wires, param_received=True)
        require(f"[{wires}]" in repr(block), "repr does not report the feature count.")
        holder = NoInputLayer(wires)
        holder.encoding = block
        require(bool(repr(holder)), "Parent module repr failed.")


run_test("External templates enforce shape (1, N)", test_external_shape_contract)
run_test("External sublayer repr reports the feature count", test_external_repr)

## 8. Invalid devices must raise direct validation errors

Invalid device values should raise `TypeError` or `ValueError`, not a later missing-attribute error.

In [15]:
def test_invalid_device_validation():
    error = require_raises(
        (TypeError, ValueError),
        lambda: NoInputLayer(1, q_device=object()),
    )
    require(
        not isinstance(error, AttributeError),
        "Invalid device handling leaked an AttributeError.",
    )


run_test("Invalid devices raise direct validation errors", test_invalid_device_validation)

[PASS] Invalid devices raise direct validation errors


## 9. Failed device updates must be atomic

If replacement construction fails, the previous device and QNode must remain installed and usable.

In [ ]:
def test_atomic_device_update():
    layer = NoInputLayer(
        1, qnode_kwargs={"interface": "torch", "diff_method": "best"}
    )
    original_device, original_qnode = layer.q_device, layer.qnode
    expected = layer()
    try:
        layer.update_qdevice(object())
    except Exception as exc:
        error = exc
    else:
        raise AssertionError("Invalid device update was accepted.")

    require(
        getattr(layer, "q_device", None) is original_device,
        "A failed update removed/replaced the original device.",
    )
    require(
        getattr(layer, "qnode", None) is original_qnode,
        "A failed update removed/replaced the original QNode.",
    )
    torch.testing.assert_close(layer(), expected)
    require(isinstance(error, (TypeError, ValueError)), f"Indirect error: {error!r}")
    original_options = copy.deepcopy(layer._qnode_kwargs)
    invalid_updates = (
        ("torchpenny.nonexistent.device", {}, {}),
        ("default.qubit", {"wires": 2}, {}),
        (qml.device("default.qubit", wires=2), {}, {}),
        ("default.qubit", {}, {"interface": "invalid_interface"}),
        ("default.qubit", {}, {"diff_method": "invalid_method"}),
    )
    for device, device_options, qnode_options in invalid_updates:
        before = copy.deepcopy((device_options, qnode_options))
        try:
            layer.update_qdevice(device, device_options, qnode_options)
        except Exception:
            pass
        else:
            raise AssertionError("Invalid device/QNode construction was accepted.")
        require(layer.q_device is original_device and layer.qnode is original_qnode,
                "Construction failure replaced working objects.")
        require(layer._qnode_kwargs == original_options, "Failed update changed settings.")
        require((device_options, qnode_options) == before, "Caller configuration was mutated.")
        torch.testing.assert_close(layer(), expected)
    for device in ("default.qubit", qml.device("default.qubit", wires=1)):
        device_options, qnode_options = {}, {"interface": "torch", "diff_method": "best"}
        layer.update_qdevice(device, device_options, qnode_options)
        require(layer.qnode.device is layer.q_device, "QNode and device disagree.")
        require(layer.qnode is not original_qnode, "Successful update retained the old QNode.")
        require(device_options == {}, "Device defaults leaked into caller settings.")
        torch.testing.assert_close(layer(), expected)


run_test("Failed device updates preserve the working layer", test_atomic_device_update)

## 10. Valid device updates must preserve QNode settings

Unless explicitly overridden, the Torch interface and differentiation method must survive a device replacement.
Repeated replacements, partial overrides, gradients, caller-owned options and failed overrides are checked.

In [ ]:
def test_device_update_preserves_qnode_settings():
    layer = ScalarLayer(
        1,
        qnode_kwargs={"interface": "torch", "diff_method": "backprop"},
    )

    layer.update_qdevice(
        "default.qubit",
        q_device_kwargs={"wires": 1},
    )

    require(
        layer.qnode.interface == "torch",
        f"Expected Torch interface, received {layer.qnode.interface!r}.",
    )
    require(
        layer.qnode.diff_method == "backprop",
        f"Expected backprop, received {layer.qnode.diff_method!r}.",
    )
    for device in (qml.device("default.qubit", wires=1), "default.qubit"):
        options = {}
        layer.update_qdevice(device, qnode_kwargs=options)
        require(options == {}, "Empty caller options were mutated.")
        require(layer.qnode.diff_method == "backprop", "Repeated replacement lost the method.")
        require(layer.qnode.interface == "torch", "Repeated replacement lost the interface.")
        x = torch.tensor([[0.2], [0.7]], dtype=torch.float64, requires_grad=True)
        output = layer(x)
        torch.testing.assert_close(output, torch.cos(x))
        output.sum().backward()
        torch.testing.assert_close(x.grad, -torch.sin(x.detach()))

    for method in ("parameter-shift", "adjoint", "best", None):
        options = {"diff_method": method, "max_diff": 2}
        before = copy.deepcopy(options)
        layer.update_qdevice("default.qubit", qnode_kwargs=options)
        require(options == before, "Override options were mutated.")
        require(layer.qnode.interface == "torch", "Partial override reset the interface.")
        require(layer.qnode.diff_method == method, "Explicit method override was ignored.")
        require(layer.qnode.execute_kwargs["max_diff"] == 2, "Additional QNode setting was lost.")
        options["diff_method"] = "invalid_method"
        layer.update_qdevice(qml.device("default.qubit", wires=1))
        require(layer.qnode.diff_method == method, "Override was not retained independently.")
        require(layer.qnode.execute_kwargs["max_diff"] == 2, "Repeated replacement reset additional settings.")
        old_device, old_qnode = layer.q_device, layer.qnode
        old_options = copy.deepcopy(layer._qnode_kwargs)
        require_raises(qml.QuantumFunctionError,
                       lambda: layer.update_qdevice("default.qubit", qnode_kwargs=options))
        require(layer.q_device is old_device and layer.qnode is old_qnode,
                "Failed override replaced the working objects.")
        require(layer._qnode_kwargs == old_options, "Failed override changed saved settings.")
        torch.testing.assert_close(layer(torch.zeros(2, 1)), torch.ones(2, 1, dtype=torch.float64))

    layer.update_qdevice("default.qubit", {"shots": 5}, {"diff_method": "best"})
    require(layer.qnode.diff_method == "best", "Finite-shot override was lost.")
    require(layer.qnode.interface == "torch", "Finite-shot update reset the interface.")
    torch.testing.assert_close(layer(torch.zeros(2, 1)), torch.ones(2, 1, dtype=torch.float64))


run_test("Device update preserves QNode configuration", test_device_update_preserves_qnode_settings)

## 11. Invalid `inner_gates()` signatures must be rejected

현재 지원하는 호출 규칙은 입력 인자 0개 또는 1개입니다.
잘못된 시그니처는 생성 또는 실행 시 `TypeError`/`ValueError`로 거부되어야 합니다.
에러 메시지의 특정 영단어를 요구하지 않습니다.

In [18]:
def test_inner_gates_signature_validation():
    class BadSignatureLayer(QLayer):
        def inner_gates(self, x, second_input):
            raise AssertionError("Invalid signature entered the circuit body.")

        def measurement(self):
            return qml.probs(wires=range(self.wires))

    require_raises((TypeError, ValueError), lambda: BadSignatureLayer(1)())


run_test("Invalid inner_gates signature is rejected", test_inner_gates_signature_validation)

[PASS] Invalid inner_gates signature is rejected


## 12. Unsupported input types must raise `TypeError`

The public boundary should reject unsupported values before uninitialized local variables or PennyLane internals produce an indirect exception.

In [19]:
def test_input_type_validation():
    layer = ScalarLayer(1, qnode_kwargs={"interface": "torch", "diff_method": "backprop"})
    check_boundary_rejection(layer, 3.14)


run_test("Unsupported scalar inputs raise TypeError before execution", test_input_type_validation)

[FAIL] Unsupported scalar inputs raise TypeError before execution
       AssertionError: Expected <class 'TypeError'>, received UnboundLocalError: cannot access local variable 'x_flat' where it is not associated with a value


## 13. Scalar measurements must preserve all batch axes

단일 expectation은 analytic/finite-shot 모두 마지막 feature 축 1을 유지합니다. 값, gradient, Linear 연결도 검증합니다.

In [ ]:
def test_scalar_measurement_shapes():
    layer = ScalarLayer(
        1, qnode_kwargs={"interface": "torch", "diff_method": "backprop"}
    )
    for shape in ((1,), (1, 1), (4, 1), (2, 4, 1)):
        x = torch.linspace(0.2, 0.8, steps=torch.Size(shape).numel(),
                           dtype=torch.float64).reshape(shape).requires_grad_()
        expected = torch.cos(x[..., 0]).unsqueeze(-1)
        output = layer(x)
        torch.testing.assert_close(output, expected)
        output.sum().backward()
        torch.testing.assert_close(x.grad, -torch.sin(x.detach()))
        head = nn.Linear(1, 2, dtype=torch.float64)
        require(head(output).shape == shape[:-1] + (2,), "Linear cannot consume the feature output.")
    class NoInputScalarLayer(QLayer):
        def inner_gates(self):
            pass

        def measurement(self):
            return qml.expval(qml.PauliZ(0))

    for shots in (None, 5, [5, 7]):
        no_input = NoInputScalarLayer(1, q_device_kwargs={"shots": shots},
                                      qnode_kwargs={"interface": "torch", "diff_method": "best"})
        result = no_input()
        partitions = result if isinstance(shots, list) else (result,)
        for output in partitions:
            require(output.shape == (1,), "Unbatched expectation lost its feature axis.")
            require(bool(torch.all(output == 1)), "Unbatched expectation changed value.")
    for shots in (1, 5, [5, 7]):
        sampled = ScalarLayer(1, q_device_kwargs={"shots": shots},
                              qnode_kwargs={"interface": "torch", "diff_method": "best"})
        for shape in ((1,), (1, 1), (2, 3, 1)):
            result = sampled(torch.zeros(shape))
            partitions = result if isinstance(shots, list) else (result,)
            for output in partitions:
                require(output.shape == shape[:-1] + (1,), "Finite-shot expectation lost its feature axis.")
                require(bool(torch.all(output == 1)), "Finite-shot expectation changed value.")


run_test("Scalar measurements preserve shape values and gradients", test_scalar_measurement_shapes)

## 14. A QLayer must use one measurement type

한 QLayer에서 probs, expval, sample 등 서로 다른 측정 종류를 혼합하지 않습니다.
혼합 측정은 device 실행 전에 ValueError로 거부합니다. device 교체 후에도 같은 규칙을 적용합니다.
단일 probs와 여러 expval은 계속 사용할 수 있으며 기존 sample 기능도 유지합니다.

In [ ]:
def test_measurement_type_policy():
    from unittest.mock import patch

    class MixedListLayer(MixedMeasurementLayer):
        def measurement(self):
            return list(super().measurement())

    class NestedMixedLayer(MixedMeasurementLayer):
        def measurement(self):
            expectation, probability = super().measurement()
            return ((expectation,), (probability,))

    class MixedSampleLayer(MixedMeasurementLayer):
        def measurement(self):
            return qml.probs(wires=[0]), qml.sample(wires=[0])

    options = {"interface": "torch", "diff_method": "best"}
    for factory in (MixedMeasurementLayer, MixedListLayer, NestedMixedLayer, MixedSampleLayer):
        for shots in (None, 5, [3, 5]):
            layer = factory(1, q_device_kwargs={"shots": shots}, qnode_kwargs=options)
            for updated in (False, True):
                if updated:
                    layer.update_qdevice("default.qubit", {"shots": shots}, options)
                with patch.object(layer.q_device, "execute",
                                  side_effect=AssertionError("Mixed measurements reached the device.")) as execute:
                    for shape in ((1,), (1, 1), (4, 1), (2, 4, 1)):
                        require_raises(ValueError, lambda: layer(torch.zeros(shape)))
                    require_raises(ValueError, lambda: layer.qnode(torch.zeros(2, 1)))
                    execute.assert_not_called()

    class MultipleExpectationLayer(QLayer):
        def inner_gates(self, x):
            qml.RY(x[:, 0], wires=0)
            qml.RY(2*x[:, 0], wires=1)

        def measurement(self):
            return self.container((qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1))))

    for container in (tuple, list):
        layer = MultipleExpectationLayer(2, qnode_kwargs=options)
        layer.container = container
        for shape in ((1,), (1, 1), (4, 1), (2, 4, 1)):
            x = torch.linspace(0.2, 0.8, steps=torch.Size(shape).numel(),
                               dtype=torch.float64).reshape(shape).requires_grad_()
            output = layer(x)
            expected = torch.stack((torch.cos(x[..., 0]), torch.cos(2*x[..., 0])), dim=-1)
            torch.testing.assert_close(output, expected)
            output.sum().backward()
            torch.testing.assert_close(x.grad, -torch.sin(x.detach())-2*torch.sin(2*x.detach()))
    probability = ProbabilityLayer(1, qnode_kwargs=options)
    torch.testing.assert_close(probability(torch.zeros(2, 1)),
                               torch.tensor([[1., 0.], [1., 0.]], dtype=torch.float64))


run_test("Mixed measurements are rejected before device execution", test_measurement_type_policy)

## 15. Finite-shot outputs must restore batch, shot, and wire axes

다른 문제로 실패하지 않도록 shot 실행과 호환되는 `diff_method="best"`를 명시합니다.
wire가 2개인 경우 출력 `(2, 3, 5, 2)`를 확인하므로
PennyLane 버전별 단일-wire squeeze 차이에 영향을 받지 않습니다.
영 입력에서는 모든 측정 비트가 0이어야 합니다.
batch 크기 1, shot 수 1, 단일 wire, shot vector도 검사합니다.
batch 축을 복원하고 측정 자체의 shape는 PennyLane의 `measurement.shape()` 규칙을 따릅니다.

In [ ]:
def test_sampling_batch_shape():
    layer = SamplingLayer(
        2,
        q_device_kwargs={"shots": 5},
        qnode_kwargs={"interface": "torch", "diff_method": "best"},
    )
    output = layer(torch.zeros(2, 3, 1))
    require(isinstance(output, torch.Tensor), "Samples must be a Torch tensor.")
    require(
        tuple(output.shape) == (2, 3, 5, 2),
        f"Expected (2, 3, 5, 2), received {tuple(output.shape)}.",
    )
    require(bool(torch.all(output == 0)), "Zero-state samples must all be zero.")
    for wires in (1, 2):
        for shots in (1, 5, [1, 5, (3, 2)]):
            layer = SamplingLayer(wires, q_device_kwargs={"shots": shots},
                                  qnode_kwargs={"interface": "torch", "diff_method": "best"})
            shot_sizes = (1, 5, 3, 3) if isinstance(shots, list) else (shots,)
            for shape in ((1,), (1, 1), (1, 1, 1), (4, 1), (2, 3, 1)):
                output = layer(torch.zeros(shape))
                if isinstance(shots, list):
                    require(isinstance(output, tuple) and len(output) == 4, "Shot partitions changed.")
                    partitions = output
                else:
                    partitions = (output,)
                for result, shot_count in zip(partitions, shot_sizes):
                    measurement_shape = qml.sample(wires=range(wires)).shape(
                        shots=shot_count, num_device_wires=wires)
                    require(result.shape == shape[:-1] + measurement_shape,
                            f"Incorrect batch/measurement shape: {result.shape}.")
                    require(bool(torch.all(result == 0)), "Samples do not match the zero state.")
    class MultipleSampleLayer(SamplingLayer):
        def measurement(self):
            return qml.sample(wires=[0, 1]), qml.sample(qml.PauliZ(0))

    layer = MultipleSampleLayer(2, q_device_kwargs={"shots": [1, 5]},
                                qnode_kwargs={"interface": "torch", "diff_method": "best"})
    result = layer(torch.zeros(2, 3, 1))
    for partition, shots in zip(result, (1, 5)):
        require(isinstance(partition, tuple), "Measurement tuple was merged.")
        sample_shape = () if shots == 1 else (shots,)
        require(partition[0].shape == (2, 3) + sample_shape + (2,), "Bit sample shape is incorrect.")
        require(partition[1].shape == (2, 3) + sample_shape, "Observable sample shape is incorrect.")
        require(bool(torch.all(partition[0] == 0)), "Bit samples changed.")
        require(bool(torch.all(partition[1] == 1)), "Observable samples changed.")


run_test("Finite-shot output preserves batch shot and wire axes", test_sampling_batch_shape)

## 16. Unsupported iterables must be rejected before execution

문자열과 숫자 리스트는 quantum 실행 이전에 거부해야 합니다.
QNode 호출 감시를 사용하므로 내부 Python 인덱싱 오류를 정상 검증으로 오인하지 않습니다.
정상 multiple-input은 비어 있지 않은 tensor tuple/list를 그대로 전달합니다.
dictionary, generator, 숫자·문자열 및 tensor가 아닌 원소를 포함한 collection은 지원하지 않습니다.

In [ ]:
def test_iterable_input_validation():
    for invalid in ("not-a-tensor", b"input", [1.0, 2.0], (), [],
                    (torch.zeros(1), 2.0), {"x": torch.zeros(1)},
                    iter([torch.zeros(1)]), {1, 2}):
        layer = ScalarLayer(1, qnode_kwargs={"interface": "torch", "diff_method": "backprop"})
        check_boundary_rejection(layer, invalid)
    class MultipleInputLayer(QLayer):
        def inner_gates(self, x):
            self.inputs_seen = x
            qml.RY(x[0][0] + x[1][0], wires=0)

        def measurement(self):
            return qml.probs(wires=[0])

    for container in (tuple, list):
        layer = MultipleInputLayer(1, qnode_kwargs={"interface": "torch", "diff_method": "backprop"})
        x = torch.tensor([0.2], dtype=torch.float64, requires_grad=True)
        y = torch.tensor([0.1], dtype=torch.float64, requires_grad=True)
        inputs = container((x, y))
        output = layer(inputs)
        require(layer.inputs_seen is inputs, "Multiple inputs were copied or flattened.")
        expected = torch.stack((torch.cos((x[0] + y[0])/2)**2,
                                torch.sin((x[0] + y[0])/2)**2))
        torch.testing.assert_close(output, expected)
        output[1].backward()
        require(x.grad is not None and y.grad is not None, "Multiple-input gradients were lost.")


run_test("Unsupported iterables are rejected before execution", test_iterable_input_validation)

## 17. Positive controls: QML execution

예외 거부만으로 통과하지 않도록 정상 확률 출력, 내부 학습 파라미터 gradient,
기본 설정의 무입력 회로도 검증합니다. 기본 interface는 `"torch"`이며
입력 없는 확률·expectation 출력, 내부 파라미터 gradient, device 교체 후 유지도 확인합니다.
사용자가 명시한 interface는 기본값으로 덮어쓰지 않습니다.

In [ ]:
def test_probability_values_and_gradients():
    layer = ProbabilityLayer(
        1, qnode_kwargs={"interface": "torch", "diff_method": "backprop"}
    )
    for shape in ((1,), (1, 1), (4, 1), (2, 4, 1)):
        x = torch.linspace(0.2, 0.8, steps=torch.Size(shape).numel(),
                           dtype=torch.float64).reshape(shape).requires_grad_()
        angle = x[..., 0]
        expected = torch.stack(
            (torch.cos(angle / 2)**2, torch.sin(angle / 2)**2), dim=-1
        )
        output = layer(x)
        torch.testing.assert_close(output, expected)
        output[..., 1].sum().backward()
        torch.testing.assert_close(x.grad, 0.5 * torch.sin(x.detach()))


def test_owned_parameter_gradient():
    layer = OwnedParameterLayer().to(dtype=torch.float64)
    output = layer()
    torch.testing.assert_close(output.sum(), torch.cos(layer.block.params).sum())
    output.sum().backward()
    torch.testing.assert_close(layer.block.params.grad, -torch.sin(layer.block.params.detach()))


def test_default_no_input_output():
    for options in (None, {}, {"diff_method": "backprop"}):
        before = copy.deepcopy(options)
        layer = NoInputLayer(1, qnode_kwargs=options)
        require(options == before, "Default interface mutated caller options.")
        for updated in (False, True):
            if updated:
                layer.update_qdevice("default.qubit")
            require(layer.qnode.interface == "torch", "Default Torch interface was not retained.")
            output = layer()
            require(isinstance(output, torch.Tensor), "Default execution must return a Torch tensor.")
            torch.testing.assert_close(
                output, torch.tensor([0.5, 0.5], dtype=output.dtype, device=output.device))

    class DefaultOwnedLayer(QLayer):
        def __init__(self):
            super().__init__(1)
            self.block = ParameterBlock(1)
            self.block.set_parameters(torch.tensor([[0.37]]))

        def inner_gates(self):
            self.block(wires=(0,))

        def measurement(self):
            return qml.expval(qml.PauliZ(0))

    layer = DefaultOwnedLayer().to(dtype=torch.float64)
    output = layer()
    torch.testing.assert_close(output, torch.cos(layer.block.params).reshape(1))
    output.sum().backward()
    torch.testing.assert_close(layer.block.params.grad, -torch.sin(layer.block.params.detach()))

    finite = NoInputLayer(1, q_device_kwargs={"shots": 10})
    output = finite()
    require(isinstance(output, torch.Tensor) and output.shape == (2,), "Finite-shot default is not Torch.")
    torch.testing.assert_close(output.sum(), torch.ones((), dtype=output.dtype))

    options = {"interface": "auto"}
    explicit = ProbabilityLayer(1, qnode_kwargs=options)
    require(explicit.qnode.interface == "auto", "Explicit interface was overwritten.")
    require(options == {"interface": "auto"}, "Explicit options were mutated.")
    require(isinstance(explicit(torch.zeros(1, 1)), torch.Tensor), "Explicit auto with Torch input failed.")


run_test("Probability outputs preserve values and input gradients", test_probability_values_and_gradients)
run_test("Owned parameters receive circuit gradients", test_owned_parameter_gradient)
run_test("Default no-input execution returns a Torch tensor", test_default_no_input_output)

## Test summary and strict completion gate

동일 이름의 최신 결과만 집계합니다. 미실행 테스트가 있으면 `NOT RUN`으로 표시하며,
`STRICT_MODE=True`에서 실패·미실행 모두 completion gate를 통과하지 못합니다.
테스트 이름을 추가하거나 바꾸면 아래 EXPECTED_TEST_NAMES도 갱신하세요.

In [ ]:
EXPECTED_TEST_NAMES = (
    "QSubLayer wire validation",
    "QLayer wire validation",
    "Caller-owned configuration remains unchanged",
    "Default configuration does not retain instance state",
    "External parameter metadata integrates with PyTorch",
    "Internal parameters support conversion and state_dict",
    "External input preserves values and gradients",
    "set_parameters safely copies values",
    "QSubLayer enforces required subclass methods",
    "Sublayer replacement keeps registries consistent",
    "Sublayer deletion removes stale state",
    "External templates enforce shape (1, N)",
    "External sublayer repr reports the feature count",
    "Invalid devices raise direct validation errors",
    "Failed device updates preserve the working layer",
    "Device update preserves QNode configuration",
    "Invalid inner_gates signature is rejected",
    "Unsupported scalar inputs raise TypeError before execution",
    "Scalar measurements preserve shape values and gradients",
    "Mixed measurements are rejected before device execution",
    "Finite-shot output preserves batch shot and wire axes",
    "Unsupported iterables are rejected before execution",
    "Probability outputs preserve values and input gradients",
    "Owned parameters receive circuit gradients",
    "Default no-input execution returns a Torch tensor",
)

missing = [name for name in EXPECTED_TEST_NAMES if name not in TEST_RESULTS]
failures = [
    (name, TEST_RESULTS[name][1])
    for name in EXPECTED_TEST_NAMES
    if name in TEST_RESULTS and not TEST_RESULTS[name][0]
]
passed = sum(TEST_RESULTS.get(name, (False, ""))[0] for name in EXPECTED_TEST_NAMES)
print(f"Result: {passed}/{len(EXPECTED_TEST_NAMES)} passed; "
      f"{len(failures)} failed; {len(missing)} not run.")
for name, details in failures:
    print(f"[FAIL] {name}: {details}")
for name in missing:
    print(f"[NOT RUN] {name}")

if STRICT_MODE:
    require(not failures and not missing, "Acceptance suite has failed or unexecuted tests.")